# Stale semantic prompt — chuẩn bị4điều kiện + kiểm tra chi phí, chưa chạy pilot
**Internet ON · T4×2 · Save & Run All.** Tối đa2000ảnh TRAIN, một target/ảnh.
Tạo cache ảnh che segmentation union và prompt trung hòa; mã hóa lại bằng CLIP frozen.
Đo4điều kiện trên128ảnh hash-first; xuất16panel kiểm tra.
Sau đó benchmark **50batch forward–backward mỗi nhánh C/T**, không optimizer.step,
kiểm tra trọng số A giữ nguyên; không tự chạy1250update hay đánh giá gain caption.
Cần checkpoint A, prompt-cache .pt, visual-cache, MSCOCO2014 **JSON+ảnh**, model promt-cache JSON gốc.
Tải official COCO annotations~253MB vào/tmp nếu chưa có archive; không tải ảnh thêm.
Đầu ra cache dự kiến khoảng2,6GB vì lưu cả feature factual và che, cộng input benchmark~300MB.
Trần30phút prep và2giờ/arm là ngân sách đề xuất; chi phí thực chưa đo.

In [ ]:
print('STALE PROMPT PREPARATION: START',flush=True)
from pathlib import Path
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
COCO_IMAGES=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/images')
PROMPT_TEXT=Path('/kaggle/input/models/ducanh2403/promt-cache/maxtext/default/1/prompt_cache.json')
ANNOTATION_ARCHIVE=Path('/tmp/stale-coco-annotations/annotations_trainval2014.zip')
OUTPUT=Path('/kaggle/working/stale_prompt_preparation_2000')
for p in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON,PROMPT_TEXT):assert p.is_file(),p
for p in (VISUAL_CACHE,COCO_IMAGES):assert p.is_dir(),p
assert not OUTPUT.exists(),'Preserve previous output; use fresh session or output path.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-stale-prompt-preparation-'))/'repo'
LOG = Path('/tmp/upt_hidden_stale-prompt-preparation.log')
PROJECT_COMMIT = 'bf997e18fafe7053ef16b8418ad87efef880c577'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'stale_data.py': '21f4962a51380b77d7c98bf6f7ecc89386b30eecb46a4d5f80fcc49e0a8fdcfe', 'run_prepare.py': '46b3e9cb6f4e5d1e4790b13dbdb976117898dc8610488fb1b6c7fa0d7291d3f8', 'profile_stale.py': '08ed37d60dda8f89a8671fcaed7dd749a10b33baf5ea1a126ee0457bf33dc1c8', 'test_stale_data.py': '54dede035da17d24bdcba5a063ffabd88efff2b304dd9e18a704753925fb2c5f', 'test_stale_runtime.py': 'a43b98378411af1853121d6207817f8b5cb97d0f4e612914a66653b202f361c3', 'diagnostic_data.py': '15c0eab863ad6500436d25372e34e3395266774838db39b2bdb8e402ff150312', 'run_diagnostic.py': '382653c64ecc7047f908ae717d5d963efa177767ad8db4774093207e286ff731', 'hidden_data.py': '9285d02c5aa7660a5656654e2efe034941bc24d6f7c300958c06fcca6f0d43c7', 'pilot_data.py': '93fa39b1de581d774417840be4d5e0b2b8d52bbf9ee53b479adad2c60a45d3d7', 'audit.py': 'cadf9d172e92e1558020520399ec6013d1141b8ab82c34821d6ba4f0dfe244b4'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocotools','ijson','h5py','numpy','Pillow'])
import torch
assert torch.cuda.device_count()==2,'Select T4x2.'
run([sys.executable,'-u','test_stale_data.py'],cwd=REPO)
run([sys.executable,'-u','test_stale_runtime.py'],cwd=REPO)


In [ ]:
cfg=dict(checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),dataset_json=str(COCO_JSON),
    images=str(COCO_IMAGES),prompt_text_json=str(PROMPT_TEXT),annotation_archive=str(ANNOTATION_ARCHIVE),output=str(OUTPUT),
    implementation_sha256=SOURCE_HASHES,project_commit=PROJECT_COMMIT)
CONFIG=Path('/tmp/stale_prompt_preparation.json');CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
PROTOCOL=OUTPUT/'protocol.json'
try:
    run([sys.executable,'-u','run_prepare.py','prepare','--config',CONFIG],cwd=REPO)
    for script,mode in [('run_prepare.py','cache'),('profile_stale.py','scores'),('profile_stale.py','benchmark')]:
        run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
            '--max_restarts=0',script,mode,'--config',PROTOCOL],cwd=REPO)
    run([sys.executable,'-u','run_prepare.py','finish','--config',PROTOCOL],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        if LOG.is_file():shutil.copyfile(LOG,OUTPUT/'notebook.log')
        for name,source in SOURCES.items():
            target=OUTPUT/'source'/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_text(source,encoding='utf-8')
        run([sys.executable,'-u','run_prepare.py','archive','--config',CONFIG],cwd=REPO)
print('KEEP ENTIRE CACHE:',OUTPUT,flush=True)
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)


Giữ toàn bộ folder cache nếu muốn dùng tiếp; ZIP review không chứa HDF5/.pt.
Gửi `stale_prompt_preparation_2000_review.zip`; xem panel.html +16ảnh để kiểm tra mask/text.
PREPARATION_COMPLETE không có nghĩa pilot đã được duyệt. Cần kết quả chi phí, panel và phản biện Argon/Astra.
Benchmark không cập nhật trọng số. Optimizer latency chưa đo; AdamW state memory chỉ ước tính, không cấp phát.
Nếu model/preprocessing parity hoặc nguồn segmentation lỗi, gửi log; không thay bằng zeroing feature cache.